# PDF toán → MinerU → Word Equation + Images + RAG

Notebook này làm đúng flow:

**User upload PDF toán**
→ **MinerU**
→ **Markdown + JSON + Images**
→ **DOCX có Heading + Table + LaTeX Equation + Images**
→ **RAG: Chunk → Embedding → FAISS → Retriever → Qwen local**

Điểm quan trọng:
- Không dùng Word làm dữ liệu trung gian cho RAG.
- Markdown/JSON của MinerU là nguồn chung cho cả DOCX và RAG.
- Công thức LaTeX được đưa vào Pandoc để chuyển sang Equation trong DOCX.
- Ảnh được lấy từ thư mục output của MinerU và giữ trong DOCX.
- Không cần API trả phí.
- Khuyến nghị dùng **Colab runtime mới**.


## 0. Cài đặt

Nếu runtime đã từng cài Docling/Pix2Text/Optimum/Transformers nhiều phiên bản, nên **Runtime → Disconnect and delete runtime** rồi mở runtime mới trước khi chạy.

In [ ]:
# MinerU + DOCX renderer + RAG
!pip install -q "mineru[all]"
!pip install -q sentence-transformers faiss-cpu pypandoc

# Pandoc để Markdown + LaTeX -> DOCX/OMML
!apt-get -qq update
!apt-get -qq install pandoc

!mineru --help
!pandoc --version


## 1. Upload PDF

In [ ]:
from google.colab import files
from pathlib import Path
import shutil, os, re, json, subprocess, textwrap

BASE = Path("/content/math_pdf_rag")
INPUT_DIR = BASE / "input"
MINERU_DIR = BASE / "mineru_output"
DOCX_DIR = BASE / "docx_output"
RAG_DIR = BASE / "rag_output"

for p in [INPUT_DIR, MINERU_DIR, DOCX_DIR, RAG_DIR]:
    p.mkdir(parents=True, exist_ok=True)

uploaded = files.upload()
pdfs = [Path(name) for name in uploaded if name.lower().endswith(".pdf")]

if not pdfs:
    raise FileNotFoundError("Chưa upload PDF.")

PDF_PATH = INPUT_DIR / pdfs[0].name
shutil.copy2(pdfs[0], PDF_PATH)

print("PDF:", PDF_PATH)


## 2. Chạy MinerU

MinerU tạo ra Markdown/JSON và tài nguyên hình ảnh. Nếu backend mặc định gặp lỗi trên Colab, dùng `pipeline`.

In [ ]:
# Chạy MinerU
# -p: input PDF
# -o: output directory
# -b pipeline: backend pipeline

!mineru -p "{PDF_PATH}" -o "{MINERU_DIR}" -b pipeline


## 3. Tìm Markdown và JSON

In [ ]:
all_files = [p for p in MINERU_DIR.rglob("*") if p.is_file()]

md_files = [p for p in all_files if p.suffix.lower() == ".md"]
json_files = [p for p in all_files if p.suffix.lower() == ".json"]
image_files = [
    p for p in all_files
    if p.suffix.lower() in [".png", ".jpg", ".jpeg", ".webp"]
]

print("Markdown:", len(md_files))
for p in md_files:
    print(" ", p)

print("\nJSON:", len(json_files))
for p in json_files[:20]:
    print(" ", p)

print("\nImages:", len(image_files))
for p in image_files[:20]:
    print(" ", p)

if not md_files:
    raise FileNotFoundError(
        "Không tìm thấy Markdown của MinerU. "
        "Hãy kiểm tra log của cell MinerU."
    )

# Chọn Markdown lớn nhất để tránh chọn file phụ nếu output có nhiều md.
MD_PATH = max(md_files, key=lambda p: p.stat().st_size)

print("\nMarkdown được chọn:", MD_PATH)


## 4. Kiểm tra Markdown và công thức

In [ ]:
markdown_text = MD_PATH.read_text(encoding="utf-8", errors="ignore")

print(markdown_text[:12000])

latex_patterns = [
    r"\\\\\(",
    r"\\\\\[",
    r"\\begin\{equation",
    r"\\frac",
    r"\\int",
    r"\\sum",
    r"\$\$",
    r"\$[^\n]+\$"
]

print("\nKiểm tra dấu hiệu LaTeX:")
for pattern in latex_patterns:
    matches = re.findall(pattern, markdown_text)
    print(pattern, "=>", len(matches))


## 5. Chuẩn hóa đường dẫn ảnh cho DOCX

MinerU Markdown thường tham chiếu ảnh tương đối. Pandoc cần tìm được các ảnh đó.

Cell dưới đây:
1. tìm thư mục chứa ảnh,
2. giữ nguyên các link ảnh nếu có thể,
3. tạo một bản Markdown làm việc với đường dẫn tuyệt đối tới ảnh.

Không thay công thức LaTeX thành text thường.


In [ ]:
# Tìm các image references trong Markdown
image_refs = re.findall(r"!\[[^\]]*\]\(([^)]+)\)", markdown_text)

print("Image references trong Markdown:", len(image_refs))
print(image_refs[:20])

# Tạo Markdown làm việc.
WORK_MD = DOCX_DIR / f"{PDF_PATH.stem}_for_docx.md"
work_text = markdown_text

# Với mỗi image path tương đối, thử resolve từ thư mục chứa MD.
for ref in sorted(set(image_refs), key=len, reverse=True):
    clean_ref = ref.strip().strip("<>")
    if clean_ref.startswith(("http://", "https://")):
        continue

    candidate = (MD_PATH.parent / clean_ref).resolve()

    if candidate.exists():
        # Pandoc chấp nhận path tuyệt đối.
        work_text = work_text.replace(ref, str(candidate))

WORK_MD.write_text(work_text, encoding="utf-8")

print("Working Markdown:", WORK_MD)


## 6. Markdown → DOCX với Equation thật

Đây là phần nâng cấp quan trọng.

Thay vì `python-docx` ghi LaTeX như text, ta dùng **Pandoc** để chuyển math trong Markdown sang định dạng equation của Word (OMML).

- Inline math: `$...$`
- Display math: `$$...$$`, `\[...\]`, hoặc các dạng LaTeX mà Markdown đầu vào của MinerU giữ lại.
- Heading → Heading Word.
- Table → Table Word.
- Image → Image Word.

Nếu một công thức đặc biệt không được Pandoc nhận dạng, công thức đó sẽ được kiểm tra ở bước validation.


In [ ]:
DOCX_PATH = DOCX_DIR / f"{PDF_PATH.stem}.docx"

cmd = [
    "pandoc",
    str(WORK_MD),
    "-o", str(DOCX_PATH),
    "--from=markdown+tex_math_dollars+tex_math_single_backslash",
    "--resource-path", str(MD_PATH.parent),
    "--standalone",
    "--toc"
]

print("Running Pandoc...")
result = subprocess.run(cmd, capture_output=True, text=True)

print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Pandoc tạo DOCX thất bại.")

print("Đã tạo DOCX:", DOCX_PATH)
print("Size:", DOCX_PATH.stat().st_size, "bytes")


## 7. Kiểm tra DOCX có tồn tại

In [ ]:
assert DOCX_PATH.exists(), "DOCX chưa được tạo."
assert DOCX_PATH.stat().st_size > 0, "DOCX rỗng."

print("DOCX OK:", DOCX_PATH)


## 8. Download DOCX

In [ ]:
files.download(str(DOCX_PATH))


## 9. Chuẩn bị dữ liệu RAG

RAG sử dụng **Markdown của MinerU**, không đọc ngược từ DOCX.

Lý do:
- Markdown giữ LaTeX.
- Markdown giữ heading/section.
- JSON có thể dùng làm metadata.
- Ảnh có thể được nối vào metadata nếu cần multimodal RAG sau này.


In [ ]:
def split_markdown_sections(md):
    sections = []
    current_heading = "Document"
    current_lines = []

    for line in md.splitlines():
        if re.match(r"^#{1,6}\s+", line):
            if current_lines:
                body = "\n".join(current_lines).strip()
                if body:
                    sections.append({
                        "heading": current_heading,
                        "text": body
                    })
                current_lines = []

            current_heading = re.sub(r"^#{1,6}\s+", "", line).strip()
        else:
            current_lines.append(line)

    if current_lines:
        body = "\n".join(current_lines).strip()
        if body:
            sections.append({
                "heading": current_heading,
                "text": body
            })

    return sections

sections = split_markdown_sections(markdown_text)

print("Sections:", len(sections))
for s in sections[:5]:
    print("\n###", s["heading"])
    print(s["text"][:500])


In [ ]:
def make_chunks(sections, max_chars=1800, overlap=250):
    chunks = []

    for sec_id, sec in enumerate(sections):
        heading = sec["heading"]
        body = sec["text"]

        if len(body) <= max_chars:
            chunks.append({
                "chunk_id": len(chunks),
                "section_id": sec_id,
                "heading": heading,
                "text": f"# {heading}\n{body}".strip()
            })
            continue

        start = 0
        while start < len(body):
            end = min(start + max_chars, len(body))
            piece = body[start:end]

            chunks.append({
                "chunk_id": len(chunks),
                "section_id": sec_id,
                "heading": heading,
                "text": f"# {heading}\n{piece}".strip()
            })

            if end >= len(body):
                break

            start = max(0, end - overlap)

    return chunks

chunks = make_chunks(sections)

print("Chunks:", len(chunks))
print(chunks[0]["text"][:2000])


## 10. Embedding pretrained model

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

EMBED_MODEL = "intfloat/multilingual-e5-base"
embedder = SentenceTransformer(EMBED_MODEL)

passages = ["passage: " + c["text"] for c in chunks]

embeddings = embedder.encode(
    passages,
    normalize_embeddings=True,
    show_progress_bar=True
)

embeddings = np.asarray(embeddings, dtype="float32")

print("Embedding shape:", embeddings.shape)


## 11. FAISS

In [ ]:
import faiss

index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)

INDEX_PATH = RAG_DIR / "index.faiss"
CHUNKS_PATH = RAG_DIR / "chunks.json"

faiss.write_index(index, str(INDEX_PATH))
CHUNKS_PATH.write_text(
    json.dumps(chunks, ensure_ascii=False, indent=2),
    encoding="utf-8"
)

print("Index:", INDEX_PATH)
print("Chunks:", CHUNKS_PATH)


## 12. Retriever

In [ ]:
def retrieve(query, top_k=5):
    query_embedding = embedder.encode(
        ["query: " + query],
        normalize_embeddings=True
    ).astype("float32")

    scores, ids = index.search(query_embedding, top_k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue

        item = dict(chunks[int(idx)])
        item["score"] = float(score)
        results.append(item)

    return results

question = "Hãy giải thích một khái niệm trong tài liệu."
results = retrieve(question, top_k=5)

for i, r in enumerate(results, 1):
    print(f"--- {i} | score={r['score']:.4f} | {r['heading']} ---")
    print(r["text"][:1200])


## 13. Optional: Qwen local để trả lời RAG

Cell này không bắt buộc. Retrieval đã hoạt động ở bước trước.

Nếu Colab có GPU, có thể dùng Qwen 2.5 3B Instruct. Nếu thiếu VRAM/RAM thì bỏ qua cell này.


In [ ]:
!pip install -q transformers accelerate


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

LLM_NAME = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)

model = AutoModelForCausalLM.from_pretrained(
    LLM_NAME,
    torch_dtype="auto",
    device_map="auto"
)

def ask_rag(question, top_k=4, max_new_tokens=512):
    retrieved = retrieve(question, top_k=top_k)

    context = "\n\n".join(
        f"[Source {i+1}]\n{r['text']}"
        for i, r in enumerate(retrieved)
    )

    user_prompt = f"""Bạn là trợ lý hỏi đáp tài liệu toán.

Chỉ sử dụng thông tin trong CONTEXT.
Nếu CONTEXT không đủ để trả lời, hãy nói rõ rằng tài liệu không cung cấp đủ thông tin.
Không tự bịa công thức.
Giữ nguyên LaTeX khi trích dẫn công thức.

CONTEXT:
{context}

QUESTION:
{question}
"""

    messages = [
        {
            "role": "system",
            "content": "Trả lời chính xác dựa trên tài liệu được truy xuất."
        },
        {
            "role": "user",
            "content": user_prompt
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        [prompt],
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    generated = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        generated,
        skip_special_tokens=True
    )

    return answer, retrieved

answer, sources = ask_rag(
    "Hãy giải thích một định nghĩa quan trọng trong tài liệu."
)

print(answer)

print("\nSOURCES:")
for s in sources:
    print(f"- {s['heading']} | score={s['score']:.4f}")


## 14. Validation cuối

Bạn cần có ít nhất:

```text
PDF
 └── input/

MinerU
 └── mineru_output/
      ├── *.md
      ├── *.json
      └── images...

Word
 └── docx_output/
      └── *.docx

RAG
 └── rag_output/
      ├── index.faiss
      └── chunks.json
```

### Luồng hoàn chỉnh

```text
                 PDF toán
                    │
                    ▼
                  MinerU
                    │
          ┌─────────┴─────────┐
          ▼                   ▼
   Markdown/JSON          Images
          │
      ┌───┴──────────────┐
      ▼                  ▼
  DOCX Renderer          RAG
      │                  │
      ▼                  ▼
 Word Equation       Chunk → E5
 + Table + Image          │
                          ▼
                        FAISS
                          │
                          ▼
                       Retriever
                          │
                          ▼
                    Qwen local LLM
```

**Kết quả:** người dùng chỉ cần đưa PDF toán vào; hệ thống có thể đồng thời xuất Word và xây knowledge base để hỏi đáp.


In [ ]:
print("===== FINAL OUTPUT =====")
print("PDF      :", PDF_PATH)
print("Markdown :", MD_PATH)
print("DOCX     :", DOCX_PATH)
print("FAISS    :", INDEX_PATH)
print("Chunks   :", CHUNKS_PATH)

print("\nExistence:")
for p in [PDF_PATH, MD_PATH, DOCX_PATH, INDEX_PATH, CHUNKS_PATH]:
    print(p.name, "=>", p.exists())
